# Fine-tune Whisper for Kinyarwanda - Stage 2 NSS
Applies elision-apostrophe normalization (`ky_helpers.py`) to training targets.




In [ ]:
from huggingface_hub import login
HF_TOKEN = input()
login(token=HF_TOKEN)

## Settings

In [ ]:
import os

LOCAL_STORAGE_DIR = '/jupyter_kernel'
BASE_DIR = os.path.join(LOCAL_STORAGE_DIR, 'trained_models')

# Stage 1 checkpoint: the KY standard-speech model
BASE_MODEL = "/jupyter_kernel/trained_models/rw_afrivoice_tune_whisper_large/best_model" # replace with your step 1 model

NSS_OUTPUT_DIR = os.path.join(
    BASE_DIR,
    'rw_afrivoice_tune_whisper_large_nss_v1.0'
)

print('Stage 1 checkpoint:', BASE_MODEL)
print('Stage 2 output dir:', NSS_OUTPUT_DIR)

if os.path.exists(NSS_OUTPUT_DIR):
    raise ValueError('NSS output dir already exists — rename or delete it first')

### Hyperparameters

In [ ]:
WHISPER_MODEL_TYPE = 'openai/whisper-large-v3'
LANGUAGE = 'sw'
TASK = 'transcribe'

LEARNING_RATE = 5e-5
MAX_STEPS = 1000
MAX_EPOCHS = 5
LR_WARMUP_STEPS = 40
LR_END = 1e-8
LR_DECAY_POWER = 4
LR_SCHEDULER_TYPE = 'polynomial'

LOGGING_STEPS = 5
SAVE_STEPS = 50
EVAL_STEPS = 50
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
MAX_GEN_LEN = 128
USE_FP16 = True
USE_BF16 = False
NUM_CHECKPOINTS_TO_STORE = 2

print(f'LR: {LEARNING_RATE}, Max steps: {MAX_STEPS}')

## Imports

In [ ]:
import datasets
from datasets import Audio
import numpy as np
import pandas as pd
import os
import re
import torch
import torchaudio
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    WhisperProcessor, WhisperForConditionalGeneration
)
import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

datasets.disable_caching()
torch.set_num_threads(1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

num_proc = min(32, os.cpu_count())
print('num_proc:', num_proc)

wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')
transcript_normalizer = BasicTextNormalizer()

### Kinyarwanda apostrophe normalization
From `ky_helpers.py` (github.com/cdl-inclusion/rwandan_languages). Strips the elision apostrophe and joins tokens: `n'abantu` -> `nabantu`. Applied to training targets, eval references, AND predictions before WER/CER scoring. Assumes text is already glyph-normalized to the canonical ASCII apostrophe.

In [ ]:
CANONICAL_APOSTROPHE = "'"
_ELISION_RE = re.compile(r"\s*" + re.escape(CANONICAL_APOSTROPHE) + r"\s*")
_WS_RE = re.compile(r"\s+")

def normalize_text(text: str) -> str:
    """ASR normalization for the elision apostrophe: remove it and join.
    Drops the canonical apostrophe and any whitespace adjacent to it, then
    collapses remaining whitespace. `n'abantu` / `n' abantu` -> `nabantu`.
    """
    if not text:
        return text
    text = _ELISION_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    return text

# Set to True for Kinyarwanda (apply to training targets AND eval metrics below).
# Set to False for English or any language without this elision pattern.
APPLY_KY_NORMALIZATION = True

# sanity check
for s in ["n'abantu", "n' abantu", "by'umwihariko", "nabantu", "n'ibindi", "Umva  n'umuntu w'inzu"]:
    print(f"{s!r:28} -> {normalize_text(s)!r}")

## Helper functions

In [ ]:
def count_trainable_parameters(model):
    params = filter(lambda p: p.requires_grad, model.parameters())
    return sum([np.prod(p.size()) for p in params])

def compute_metrics(pred, apply_ky_normalization=True):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_strs = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_strs = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    print(f'\n=== STEP {trainer.state.global_step} SAMPLE PREDICTIONS ===')
    for i, (ref, pred_str) in enumerate(zip(label_strs[:10], pred_strs[:10])):
        print(f'[{i}] ref:  {ref}')
        print(f'[{i}] pred: {pred_str}')
        print()

    # Fix: normalize_text() (apostrophe join) must run BEFORE transcript_normalizer()
    # (BasicTextNormalizer), since the latter turns the apostrophe into a space --
    # if it ran first, normalize_text() would find no apostrophe left to join.
    wers, cers = [], []
    for p, l in zip(pred_strs, label_strs):
        if apply_ky_normalization and APPLY_KY_NORMALIZATION:
            p = transcript_normalizer(normalize_text(p))
            l = transcript_normalizer(normalize_text(l))
        else:
            p = transcript_normalizer(p)
            l = transcript_normalizer(l)
        wers.append(wer_metric.compute(predictions=[p], references=[l]))
        cers.append(cer_metric.compute(predictions=[p], references=[l]))
    wer = np.mean([min(1.0, x) for x in wers])
    cer = np.mean([min(1.0, x) for x in cers])
    print('wer:', wer, 'cer:', cer)
    return {'wer': wer, 'cer': cer}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

## Load processor and feature extractor

In [ ]:
processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_TYPE, language=LANGUAGE, task=TASK)

def augment_audio(audio_array, sampling_rate):
    audio = audio_array.astype(np.float32)

    if np.random.rand() < 0.5:
        noise_level = np.random.uniform(0.001, 0.015)
        audio = audio + noise_level * np.random.randn(len(audio))

    if np.random.rand() < 0.5:
        gain = np.random.uniform(0.7, 1.3)
        audio = audio * gain

    if np.random.rand() < 0.5:
        speed_factor = np.random.uniform(0.9, 1.1)
        audio_tensor = torch.tensor(audio).unsqueeze(0)
        resampled = torchaudio.functional.resample(
            audio_tensor,
            orig_freq=int(sampling_rate * speed_factor),
            new_freq=sampling_rate
        )
        audio = resampled.squeeze(0).numpy()

    return np.clip(audio, -1.0, 1.0)

def prepare_features(example, augment=False):
    audio_array = example['audio']['array']
    sampling_rate = example['audio']['sampling_rate']
    if augment:
        audio_array = augment_audio(audio_array, sampling_rate)
    example['input_features'] = processor.feature_extractor(
        audio_array, sampling_rate=sampling_rate
    ).input_features[0]
    transcription = normalize_text(example['transcription'])  # elision-strip for KY
    example['labels'] = processor.tokenizer.encode(transcription)
    example['token_length'] = len(example['labels'])
    return example

## Load datasets

In [ ]:
# AfriVoice (standard speech) dev and test — stream -> materialize (no multiprocessing, no cache)

from datasets import Dataset

def load_and_process_split(split, seed=42):
    streaming_ds = (
        datasets.load_dataset(
            'ElizabethMwangi/kinyarwanda_afrivoice_all_domains_v0.2',
            split=split,
            streaming=True,
            token=HF_TOKEN,
        )
        .cast_column('audio', Audio(sampling_rate=16000))
        .shuffle(seed=seed, buffer_size=1000)
        .map(prepare_features, remove_columns=['audio'])
        .filter(lambda x: x['token_length'] <= 448)
    )
    return Dataset.from_generator(lambda: (yield from streaming_ds))

ds_dev = load_and_process_split('validation')
print('dev set:', ds_dev)

ds_test = load_and_process_split('test')
print('test set:', ds_test)

In [ ]:
ds_nss_train_raw, ds_nss_val_raw, ds_test_nss_raw = [
    datasets.load_dataset('cdli/rwandan_kinyarwanda_nonstandard_speech_v1.0', split=split, token=HF_TOKEN)
    for split in ['train', 'validation', 'test']
]

def process(ds, augment=False):
    return ds.filter(lambda x: x['audio_length'] <= 30).shuffle(seed=42).flatten_indices().map(
        lambda x: prepare_features(x, augment=augment),
        remove_columns=['audio'], writer_batch_size=1, num_proc=1
    ).filter(lambda x: x['token_length'] <= 448)

ds_nss_train = process(ds_nss_train_raw, augment=True)
ds_nss_val, ds_test_nss = [process(ds) for ds in [ds_nss_val_raw, ds_test_nss_raw]]

print('NSS train:', ds_nss_train)
print('NSS val:', ds_nss_val)
print('NSS test:', ds_test_nss)

In [ ]:
import os
from datasets import load_from_disk

NSS_CACHE = "/jupyter_kernel/nss_processed_ky"
os.makedirs(NSS_CACHE, exist_ok=True)
ds_nss_train.save_to_disk(f"{NSS_CACHE}/train")
ds_nss_val.save_to_disk(f"{NSS_CACHE}/validation")
ds_test_nss.save_to_disk(f"{NSS_CACHE}/test")
print("Saved to cache.")

In [ ]:
from datasets import load_from_disk

NSS_CACHE = "/jupyter_kernel/nss_processed_ky"
ds_nss_train = load_from_disk(f"{NSS_CACHE}/train")
ds_nss_val   = load_from_disk(f"{NSS_CACHE}/validation")
ds_test_nss  = load_from_disk(f"{NSS_CACHE}/test")
print('NSS train:', ds_nss_train)
print('NSS val:',   ds_nss_val)
print('NSS test:',  ds_test_nss)

### Sample check

In [ ]:
sample = ds_nss_train[0]
print('transcription:', processor.tokenizer.decode(sample['labels'], skip_special_tokens=True))
print('token_length:', sample['token_length'])
print('input_features shape:', len(sample['input_features']), 'x', len(sample['input_features'][0]))

## Load Stage 1 checkpoint

In [ ]:
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
_ = base_model.to(device)
base_model.config.apply_spec_augment = True
base_model.generation_config.language = LANGUAGE
base_model.generation_config.task = TASK
base_model.generation_config.forced_decoder_ids = None
base_model.config.forced_decoder_ids = None
base_model.config.use_cache = False
base_model.model.encoder.requires_grad_(True)
base_model.model.decoder.requires_grad_(True)
base_model.proj_out.requires_grad_(True)
print('trainable params:', count_trainable_parameters(base_model))

### Verify trainable parameters

In [ ]:
print('encoder trainable:', all(p.requires_grad for p in base_model.model.encoder.parameters()))
print('decoder trainable:', all(p.requires_grad for p in base_model.model.decoder.parameters()))
print('proj_out trainable:', all(p.requires_grad for p in base_model.proj_out.parameters()))

## Stage 2 — NSS fine-tuning

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=NSS_OUTPUT_DIR,
    logging_dir=os.path.join(NSS_OUTPUT_DIR, 'logs'),
    logging_steps=LOGGING_STEPS,
    report_to=['tensorboard'],
    include_num_input_tokens_seen=True,
    fp16=USE_FP16,
    bf16=USE_BF16,
    push_to_hub=False,
    remove_unused_columns=False,
    num_train_epochs=MAX_EPOCHS,
    max_steps=MAX_STEPS,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    per_device_train_batch_size=BATCH_SIZE,
    eval_on_start=True,
    predict_with_generate=True,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    eval_steps=EVAL_STEPS,
    eval_strategy='steps',
    generation_max_length=MAX_GEN_LEN,
    metric_for_best_model='wer',
    greater_is_better=False,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    lr_scheduler_kwargs={'lr_end': LR_END, 'power': LR_DECAY_POWER},
    learning_rate=LEARNING_RATE,
    warmup_steps=LR_WARMUP_STEPS,
    save_steps=SAVE_STEPS,
    save_strategy='steps',
    save_total_limit=NUM_CHECKPOINTS_TO_STORE,
    load_best_model_at_end=True,
)

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=base_model.config.decoder_start_token_id,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=base_model,
    train_dataset=ds_nss_train,
    eval_dataset=ds_nss_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  # normalized (apostrophe-stripped) by default
    processing_class=processor,
)

trainer.train()
#trainer.train(resume_from_checkpoint=True)

## Save best model

In [ ]:
best_model_dir = os.path.join(NSS_OUTPUT_DIR, 'best_model')
trainer.save_model(best_model_dir)
processor.save_pretrained(best_model_dir)
print('Saved to:', best_model_dir)